# MP Forward / Inverse Demo

This notebook demonstrates: population law -> MP forward density -> MP inverse estimate.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'mpdiff').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root (expected pyproject.toml and src/mpdiff).')
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt

from mpdiff.config.loader import load_config
from mpdiff.experiments.common import build_population_spectrum, resolve_aspect_ratio
from mpdiff.spectral.grids import make_linear_grid
from mpdiff.spectral.transforms import mp_forward_transform
from mpdiff.spectral.inverse import invert_mp_density
from mpdiff.utils.random import make_rng

cfg = load_config(PROJECT_ROOT / 'configs/constant_diag_dirac.yaml')
rng = make_rng(cfg.global_settings.seed)
population = build_population_spectrum(cfg, rng)
c = resolve_aspect_ratio(cfg)
grid = make_linear_grid(cfg.mp_forward.grid_min, cfg.mp_forward.grid_max, cfg.mp_forward.num_points)

observed = mp_forward_transform(population, c, grid, eta=cfg.mp_forward.eta, tol=cfg.mp_forward.tol, max_iter=cfg.mp_forward.max_iter, damping=cfg.mp_forward.damping)
inv = invert_mp_density(observed, c, cfg.mp_inverse, cfg.mp_forward)

plt.figure(figsize=(8,4))
plt.plot(observed.grid, observed.density, label='forward observed')
plt.plot(inv.reconstructed_observed.grid, inv.reconstructed_observed.density, label='inverse reconstructed')
plt.legend(frameon=False)
plt.title(f'Inverse method: {cfg.mp_inverse.method}')
plt.show()

print('population mean:', population.mean())
print('estimated mean:', inv.estimated_population.mean())
print('diagnostics:', inv.diagnostics)